In [10]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [11]:
load_dotenv()
llm=ChatGroq(model='llama-3.1-8b-instant')


In [12]:
class BLogstate(TypedDict):
    title:str
    outline:str
    blog:str

In [13]:
graph=StateGraph(BLogstate)

In [14]:
def create_outline(state:BLogstate)->BLogstate:
    title=state['title']
    prompt=f'generate the detailed outline for blog on following topic{title}'
    response=llm.invoke(prompt)
    state['outline']=response.content
    return state

In [15]:
def create_blog(state:BLogstate)->BLogstate:
    title=state['title']
    outline=state['outline']
    prompt=f'write a detailed blog on following topic -{title} using following outline{outline}'
    response=llm.invoke(prompt)
    state['blog']=response.content
    return state

In [16]:
# add node
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)
workflow=graph.compile()

In [18]:
initial_state={'title':'rise of india and ai'}
final=workflow.invoke(initial_state)
final['outline']
final['blog']

'**The Rise of India and AI: A Perfect Storm of Innovation and Growth**\n\n**I. Introduction**\n\nIndia, the world\'s third-largest economy, is poised to become a global leader in Artificial Intelligence (AI). The country\'s large talent pool, growing economy, and supportive government policies have created a perfect storm of innovation and growth. In this blog, we will explore the rise of India and AI, highlighting the country\'s AI ecosystem, talent and education, government support, AI applications, challenges, and opportunities.\n\nIndia\'s growing economy and technological advancements have created a fertile ground for AI development. The country\'s IT industry, valued at over $150 billion, is expected to grow to $350 billion by 2025. AI has the potential to drive innovation and growth in various sectors, including healthcare, finance, education, and transportation. The Indian government has recognized the importance of AI and has launched several initiatives to promote its develo